In [ ]:
import torch
import os

dir_loaders = 'dataloaders'
rock_loader = torch.load(os.path.join(dir_loaders, 'rock_dataloader_zeros.pt'), weights_only=False)

all_x = []
for x, _ in rock_loader:
    all_x.append(x)
all_x = torch.cat(all_x, dim=0)

all_x.shape

torch.Size([592, 1, 8, 400])

In [7]:
# We need to convert [8, 400] part to entropy, wavelet energy, etc.
import torch
import numpy as np
import pywt
from scipy.signal import welch

# ---------- Feature functions ----------
def rms(signal):
    return np.sqrt(np.mean(signal**2))

def waveform_length(signal):
    return np.sum(np.abs(np.diff(signal)))

def zero_crossings(signal, threshold=0.01):
    # Count sign changes with threshold
    count = np.sum(((signal[:-1] * signal[1:]) < 0) &
                   (np.abs(signal[:-1] - signal[1:]) > threshold))
    return count

def median_freq(signal, fs=1000):  # <-- adjust fs if known
    freqs, psd = welch(signal, fs=fs, nperseg=min(256, len(signal)))
    cumulative = np.cumsum(psd)
    cutoff = cumulative[-1] / 2
    return freqs[np.where(cumulative >= cutoff)[0][0]]

def sample_entropy(signal, m=2, r=0.2):
    # Approximate sample entropy
    N = len(signal)
    r *= np.std(signal)
    
    def _phi(m):
        x = np.array([signal[i:i+m] for i in range(N - m + 1)])
        C = np.sum(np.sum(np.abs(x[:,None] - x[None,:]), axis=2) <= r, axis=0) - 1
        return np.sum(C) / ((N - m + 1) * (N - m))
    
    return -np.log(_phi(m+1) / _phi(m) + 1e-10)

def shannon_entropy(signal, bins=30):
    hist, _ = np.histogram(signal, bins=bins, density=True)
    hist = hist[hist > 0]
    return -np.sum(hist * np.log2(hist))

def wavelet_energy(signal, wavelet='db4', level=3):
    coeffs = pywt.wavedec(signal, wavelet, level=level)
    energies = [np.sum(np.square(c)) for c in coeffs]
    return np.sum(energies)

# ---------- Main extractor ----------
def extract_emg_features(emg_tensor, fs=1000):
    """
    emg_tensor: torch.Size([samples, 1, channels, timesteps])
    Returns: numpy array of shape [samples, channels * n_features]
    """
    emg_np = emg_tensor.squeeze(1).numpy()  # shape: [samples, channels, timesteps]
    n_samples, n_channels, n_timesteps = emg_np.shape
    
    features = []
    for s in range(n_samples):
        sample_feats = []
        for ch in range(n_channels):
            sig = emg_np[s, ch, :]
            
            feats = [
                rms(sig),
                waveform_length(sig),
                zero_crossings(sig),
                median_freq(sig, fs),
                sample_entropy(sig),
                shannon_entropy(sig),
                wavelet_energy(sig),
            ]
            sample_feats.extend(feats)
        features.append(sample_feats)
    
    return np.array(features)

# ---------- Example usage ----------
if __name__ == "__main__":
    dir_loaders = 'dataloaders'
    types = ['rock', 'paper', 'scissor']

    feature_data = {}

    for _type in types:
        print(f'Processing {_type} data...')
        loader = torch.load(os.path.join(dir_loaders, f'{_type}_dataloader_zeros.pt'), weights_only=False)

        # Load data
        all_x = []
        for x, _ in loader:
            all_x.append(x)
        all_x = torch.cat(all_x, dim=0)

        X_feats = extract_emg_features(all_x, fs=100)
        print("Feature matrix shape:", X_feats.shape)

        feature_data[_type] = X_feats

feature_data


Processing rock data...
Feature matrix shape: (592, 56)
Processing paper data...
Feature matrix shape: (592, 56)
Processing scissor data...
Feature matrix shape: (592, 56)


{'rock': array([[ 1.65557861e+00,  7.62939636e+02,  1.90000000e+02, ...,
          1.98789347e-01,  7.07943168e+00,  1.78644302e+02],
        [ 5.85054040e-01,  1.61002258e+02,  1.59000000e+02, ...,
          3.42270333e-01, -6.93763133e-01,  8.22572021e+01],
        [ 1.22769022e+00,  4.57284698e+02,  1.67000000e+02, ...,
          3.48711483e-01,  5.36920610e+00,  7.58807007e+02],
        ...,
        [ 3.37771177e-01,  1.25588974e+02,  1.58000000e+02, ...,
          2.29842313e-01, -6.68400263e-01,  9.48888245e+01],
        [ 5.47873318e-01,  1.69224503e+02,  1.68000000e+02, ...,
          5.20111504e-01, -1.15423031e+01,  2.80630264e+01],
        [ 2.44737387e+00,  9.80854797e+02,  1.83000000e+02, ...,
          2.15463747e-01,  7.04785799e+00,  1.24743726e+03]],
       shape=(592, 56)),
 'paper': array([[7.64267087e-01, 1.92710144e+02, 1.69000000e+02, ...,
         7.63661988e-01, 9.55019107e-01, 8.65253296e+01],
        [2.73240000e-01, 1.05848587e+02, 1.92000000e+02, ...,
      